In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Building Knowledge Graphs with Gemini

<table align="left">
  <tr>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fuse-cases%2Fknowledge-graph%2Fknowledge_graph_generation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  </tr>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/use-cases/knowledge-graph/knowledge_graph_generation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| Author                                           |
| ------------------------------------------------ |
| [Laurent Picard](https://github.com/PicardParis) |


---

## ✨ Overview


![intro image](https://github.com/PicardParis/cherry-on-py-pics/raw/main/misc/Building-Knowledge-Graphs-with-Gemini.png)

In this exploration, we'll see how to turn raw, unstructured documents into structured knowledge graphs using Gemini. We'll start prototyping to build our intuition, optimize our prompts and outputs, and finally scale up to process entire books or dense legal contracts. By the end, we'll even visualize the extracted book narrative or contractual network graphs!


---

## 🔥 Challenge


Can we extract knowledge from documents with just the following?

- 1 document
- 1 prompt
- 1 request

Let's try with Gemini…


---

## 🏁 Setup


### 🐍 Python packages


We'll use the following packages:

- `google-genai`: the [Google Gen AI Python SDK](https://pypi.org/project/google-genai) lets us call Gemini with a few lines of code
- `networkx` for graph management

We'll also use these packages:

- `tenacity` for request management (dependency of `google-genai`)
- `matplotlib` and `pillow` for data visualization (dependencies of `networkx`)


In [ ]:
%pip install --quiet "google-genai>=1.68.0" "networkx[default]"

---

### 🤝 Gemini API


To use the Gemini API, we have two main options:

1. Via **Agent Platform** (formerly known as Vertex AI) with a Google Cloud project
2. Via **Google AI Studio** with a Gemini API key

The Google Gen AI SDK provides a unified interface to these APIs and we can use environment variables for the configuration.

**🛠️ Option 1 - Gemini API via Agent Platform**

Requirements:

- A Google Cloud project
- The Agent Platform API must be enabled for this project: ▶️ [Enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com,storage-component.googleapis.com)

Gen AI SDK environment variables:

- `GOOGLE_GENAI_USE_VERTEXAI="True"`
- `GOOGLE_CLOUD_PROJECT="<PROJECT_ID>"`
- `GOOGLE_CLOUD_LOCATION="<LOCATION>"`

> 💡 For preview models, the location must be set to `global`. For generally available models, we can choose the closest location among the [Google model endpoint locations](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/locations#google-models).

> ℹ️ Learn more about [setting up a project and a development environment](https://docs.cloud.google.com/vertex-ai/docs/start/cloud-environment).

**🛠️ Option 2 - Gemini API via Google AI Studio**

Requirement:

- A Gemini API key

Gen AI SDK environment variables:

- `GOOGLE_GENAI_USE_VERTEXAI="False"`
- `GOOGLE_API_KEY="<API_KEY>"`

> ℹ️ Learn more about [getting a Gemini API key from Google AI Studio](https://aistudio.google.com/app/apikey).


💡 You can store your environment configuration outside of the source code:

| Environment      | Method                                                      |
| ---------------- | ----------------------------------------------------------- |
| IDE              | `.env` file (or equivalent)                                 |
| Colab            | Colab Secrets (🗝️ icon in left panel, see code below)       |
| Colab Enterprise | Google Cloud project and location are automatically defined |
| Workbench        | Google Cloud project and location are automatically defined |


Define the following environment detection functions. You can also define your configuration manually if needed.


In [ ]:
# @title {display-mode: "form"}
import os
import sys
from collections.abc import Callable

from google import genai

# Manual setup (leave unchanged if setup is environment-defined)

# @markdown **Which API: Agent Platform (formerly known as Vertex AI) or Google AI Studio?**
GOOGLE_GENAI_USE_VERTEXAI = True  # @param {type: "boolean"}

# @markdown **Option A - Google Cloud project [+location]**
GOOGLE_CLOUD_PROJECT = ""  # @param {type: "string"}
GOOGLE_CLOUD_LOCATION = "global"  # @param {type: "string"}

# @markdown **Option B - Google AI Studio API key**
GOOGLE_API_KEY = ""  # @param {type: "string"}


def check_environment() -> bool:
    check_colab_user_authentication()
    return check_manual_setup() or check_vertex_ai() or check_colab() or check_local()


def check_manual_setup() -> bool:
    return check_define_env_vars(
        GOOGLE_GENAI_USE_VERTEXAI,
        GOOGLE_CLOUD_PROJECT.strip(),  # Might have been pasted with line return
        GOOGLE_CLOUD_LOCATION,
        GOOGLE_API_KEY,
    )


def check_vertex_ai() -> bool:
    # Workbench and Colab Enterprise
    match os.getenv("VERTEX_PRODUCT", ""):
        case "WORKBENCH_INSTANCE":
            pass
        case "COLAB_ENTERPRISE":
            if not running_in_colab_env():
                return False
        case _:
            return False

    return check_define_env_vars(
        True,
        os.getenv("GOOGLE_CLOUD_PROJECT", ""),
        os.getenv("GOOGLE_CLOUD_REGION", ""),
        "",
    )


def check_colab() -> bool:
    if not running_in_colab_env():
        return False

    # Colab Enterprise was checked before, so this is Colab only
    from google.colab import auth as colab_auth  # type: ignore

    colab_auth.authenticate_user()

    # Use Colab Secrets (🗝️ icon in left panel) to store the environment variables
    # Secrets are private, visible only to you and the notebooks that you select
    # - Agent Platform: Store your settings as secrets
    # - Google AI: Directly import your Gemini API key from the UI
    vertexai, project, location, api_key = get_vars(get_colab_secret)

    return check_define_env_vars(vertexai, project, location, api_key)


def check_local() -> bool:
    vertexai, project, location, api_key = get_vars(os.getenv)

    return check_define_env_vars(vertexai, project, location, api_key)


def running_in_colab_env() -> bool:
    # Colab or Colab Enterprise
    return "google.colab" in sys.modules


def check_colab_user_authentication() -> None:
    if running_in_colab_env():
        from google.colab import auth as colab_auth  # type: ignore

        colab_auth.authenticate_user()


def get_colab_secret(secret_name: str, default: str) -> str:
    from google.colab import errors, userdata  # type: ignore

    try:
        return userdata.get(secret_name)
    except errors.SecretNotFoundError:
        return default


def disable_colab_cell_scrollbar() -> None:
    if running_in_colab_env():
        from google.colab import output  # type: ignore

        output.no_vertical_scroll()


def get_vars(getenv: Callable[[str, str], str]) -> tuple[bool, str, str, str]:
    # Limit getenv calls to the minimum (may trigger UI confirmation for secret access)
    vertexai_str = getenv("GOOGLE_GENAI_USE_VERTEXAI", "")
    if vertexai_str:
        vertexai = vertexai_str.lower() in ["true", "1"]
    else:
        vertexai = bool(getenv("GOOGLE_CLOUD_PROJECT", ""))

    project = getenv("GOOGLE_CLOUD_PROJECT", "") if vertexai else ""
    location = getenv("GOOGLE_CLOUD_LOCATION", "") if project else ""
    api_key = getenv("GOOGLE_API_KEY", "") if not project else ""

    return vertexai, project, location, api_key


def check_define_env_vars(
    vertexai: bool,
    project: str,
    location: str,
    api_key: str,
) -> bool:
    match (vertexai, bool(project), bool(location), bool(api_key)):
        case (True, True, _, _):
            # Agent Platform - Google Cloud project [+location]
            location = location or "global"
            define_env_vars(vertexai, project, location, "")
        case (True, False, _, True):
            # Agent Platform - API key
            define_env_vars(vertexai, "", "", api_key)
        case (False, _, _, True):
            # Google AI Studio - API key
            define_env_vars(vertexai, "", "", api_key)
        case _:
            return False

    return True


def define_env_vars(vertexai: bool, project: str, location: str, api_key: str) -> None:
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = str(vertexai)
    os.environ["GOOGLE_CLOUD_PROJECT"] = project
    os.environ["GOOGLE_CLOUD_LOCATION"] = location
    os.environ["GOOGLE_API_KEY"] = api_key


def check_configuration(client: genai.Client) -> None:
    service = "Agent Platform" if client.vertexai else "Google AI Studio"
    print(f"✅ Using the {service} API", end="")

    if client._api_client.project:
        print(f' with project "{client._api_client.project[:7]}…"', end="")
        print(f' in location "{client._api_client.location}"')
    elif client._api_client.api_key:
        api_key = client._api_client.api_key
        print(f' with API key "{api_key[:5]}…{api_key[-5:]}"', end="")
        print(f" (in case of error, make sure it was created for {service})")


print("✅ Environment functions defined")

---

### 🤖 Gen AI SDK


To send Gemini requests, create a `google.genai` client:


In [ ]:
from google import genai

check_environment()

client = genai.Client()

check_configuration(client)

---

### 🔣 Input data


We'll need data to test our solution. Classic books are a good source of text of varying sizes and languages.

> ⚠️ LLMs are trained on general knowledge, which becomes part of their "long-term memory". To avoid generating memorized information, we must remember to be explicit about only using (extracting from) the provided text.


Let's define a few data sources and helpers:


In [ ]:
# @title {display-mode: "form"}
import mimetypes
from collections.abc import Iterator
from enum import Enum
from pathlib import Path

from google.genai.types import Part

GOOGLE_CLOUD_STORAGE_PREFIX = "gs://"
HTTPS_PREFIX = "https://"
FILE_PREFIX = "file://"
LOCAL_FOLDER = "./"


class Source(Enum):
    def yield_contents(self) -> Iterator[Part]:
        file_uri = self.value
        if not client.vertexai:
            file_uri = convert_to_https_url_if_cloud_storage_uri(file_uri)
        mime_type, _ = mimetypes.guess_type(file_uri)
        assert mime_type is not None, f"❌ Could not determine mime type: {file_uri=}"

        if file_uri.startswith((GOOGLE_CLOUD_STORAGE_PREFIX, HTTPS_PREFIX)):
            yield Part.from_uri(file_uri=file_uri, mime_type=mime_type)
            return

        if file_uri.startswith(FILE_PREFIX):
            file = Path(file_uri.removeprefix(FILE_PREFIX))
            assert file.exists(), f"❌ File does not exist: {file=}"
            if mime_type == "text/plain":
                yield Part.from_text(text=file.read_text(encoding="utf-8"))
            else:
                yield Part.from_bytes(data=file.read_bytes(), mime_type=mime_type)
            return


def convert_to_https_url_if_cloud_storage_uri(uri: str) -> str:
    return (
        f"{HTTPS_PREFIX}storage.googleapis.com/{uri.removeprefix(GOOGLE_CLOUD_STORAGE_PREFIX)}"
        if uri.startswith(GOOGLE_CLOUD_STORAGE_PREFIX)
        else uri
    )


def local_file(filename: str) -> str:
    return f"{FILE_PREFIX}{LOCAL_FOLDER}{filename}"


# You can find public domain books on Project Gutenberg: https://gutenberg.org/ebooks
def project_gutenberg_txt_url(id: int) -> str:
    return f"{HTTPS_PREFIX}gutenberg.org/cache/epub/{id}/pg{id}.txt"


class Classic(Source):
    en_hugo_les_misérables = project_gutenberg_txt_url(135)
    en_dumas_count_of_monte_cristo = project_gutenberg_txt_url(1184)
    fr_zola_thérèse_raquin = project_gutenberg_txt_url(7461)
    fr_dumas_trois_mousquetaires = project_gutenberg_txt_url(13951)
    fr_dumas_vingt_ans_après = project_gutenberg_txt_url(13952)
    fr_dumas_comte_de_monte_cristo_1 = project_gutenberg_txt_url(17989)
    fr_dumas_comte_de_monte_cristo_2 = project_gutenberg_txt_url(17990)
    fr_dumas_comte_de_monte_cristo_3 = project_gutenberg_txt_url(17991)
    fr_dumas_comte_de_monte_cristo_4 = project_gutenberg_txt_url(17992)


class Document(Source):
    en_pharma_dev_agreement = "gs://cloud-samples-data/documentai/ContractDocAI/CUAD_v1/Part_I/Development/PhasebioPharmaceuticalsInc_20200330_10-K_EX-10.21_12086810_EX-10.21_Development Agreement.pdf"


class Collection(Source):
    fr_comte_de_monte_cristo = [
        Classic.fr_dumas_comte_de_monte_cristo_1,
        Classic.fr_dumas_comte_de_monte_cristo_2,
        Classic.fr_dumas_comte_de_monte_cristo_3,
        Classic.fr_dumas_comte_de_monte_cristo_4,
    ]
    fr_trois_mousquetaires_vingt_ans_après = [
        Classic.fr_dumas_trois_mousquetaires,
        Classic.fr_dumas_vingt_ans_après,
    ]

    def yield_contents(self) -> Iterator[Part]:
        for text in self.value:
            yield from text.yield_contents()


print("✅ Data helpers defined")

---

### 🧠 Gemini model


Gemini comes in different [versions](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models#gemini-models) and sizes (Flash-Lite, Flash, and Pro).

Let's get started with Gemini 3.1 Flash-Lite, as it offers high performance with low latency and a very high output speed:

- `GEMINI_3_1_FLASH_LITE = "gemini-3.1-flash-lite-preview"`


---

### ⚙️ Gemini configuration


Gemini can be used in different ways, ranging from factual to creative modes. The problem we're trying to solve corresponds to a **data extraction** use case. We want results as factual and deterministic as possible. For this, we can change the [content generation parameters](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/multimodal/content-generation-parameters).

We'll set the `temperature`, `top_p`, and `seed` parameters to minimize randomness:

- `temperature=0.0`
- `top_p=0.0`
- `seed=42` (arbitrary fixed value)


---

### 🛠️ Helpers


Now, let's add core helper classes and functions:


In [ ]:
# @title {display-mode: "form"}
from enum import StrEnum, auto

import IPython.display
import tenacity
from google.genai.errors import ClientError
from google.genai.types import (
    FinishReason,
    GenerateContentConfig,
    GenerateContentResponse,
    ThinkingConfig,
    ThinkingLevel,
)


class Model(Enum):
    # Generally Available (GA)
    GEMINI_2_5_FLASH_LITE = "gemini-2.5-flash-lite"
    GEMINI_2_5_FLASH = "gemini-2.5-flash"
    GEMINI_2_5_PRO = "gemini-2.5-pro"
    # Preview
    GEMINI_3_1_FLASH_LITE = "gemini-3.1-flash-lite-preview"
    GEMINI_3_FLASH = "gemini-3-flash-preview"
    GEMINI_3_1_PRO = "gemini-3.1-pro-preview"
    # Default model
    DEFAULT = GEMINI_3_1_FLASH_LITE


# Default configuration for more deterministic outputs
DEFAULT_CONFIG = GenerateContentConfig(
    temperature=0.0,
    top_p=0.0,
    seed=42,  # Arbitrary fixed value
)


class ShowAs(StrEnum):
    DONT_SHOW = auto()
    TEXT = auto()
    MARKDOWN = auto()


def generate_content(
    prompt: str,
    source: Source | str | None = None,
    *,
    model: Model | None = None,
    config: GenerateContentConfig | None = None,
    system_instruction: str | None = None,
    show_prompt: ShowAs = ShowAs.DONT_SHOW,
    show_response: ShowAs = ShowAs.MARKDOWN,
    only_show_prompt: bool = False,
    return_response: bool = False,
) -> GenerateContentResponse | None:
    model = model or Model.DEFAULT
    model_id = model.value
    prompt_contents = get_prompt_contents(prompt, source, show_prompt, only_show_prompt)
    if only_show_prompt:
        return None
    config = config or get_generate_content_config(model, system_instruction)
    client = check_client_for_model(model)

    response = None
    display_request_header(model_id, source)
    for attempt in get_retrier():
        with attempt:
            response = client.models.generate_content(
                model=model_id,
                contents=prompt_contents,  # type: ignore
                config=config,
            )
            display_response_info(response)
            display_response(response, show_response)

    return response if return_response else None


def get_prompt_contents(
    prompt: str,
    source: Source | str | None,
    show_prompt: ShowAs,
    only_show_prompt: bool,
) -> list[str | Part]:
    def yield_prompt_contents() -> Iterator[str | Part]:
        if source:
            yield "==Start of input data==\n"
            if isinstance(source, str):
                yield f"{source.strip()}\n"
            else:
                yield from source.yield_contents()
            yield "==End of input data==\n"
        yield prompt.strip()

    prompt_contents = list(yield_prompt_contents())
    display_prompt(prompt_contents, show_prompt, only_show_prompt)

    return prompt_contents


def get_generate_content_config(
    model: Model,
    system_instruction: str | None = None,
) -> GenerateContentConfig:
    thinking_config = get_thinking_config_for_model(model)

    return GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=DEFAULT_CONFIG.temperature,
        top_p=DEFAULT_CONFIG.top_p,
        seed=DEFAULT_CONFIG.seed,
        thinking_config=thinking_config,
    )


def get_thinking_config_for_model(model: Model) -> ThinkingConfig | None:
    # Use minimal thinking configurations since our prompt will directly provide a chain of thoughts
    match model:
        case Model.GEMINI_2_5_FLASH:
            return ThinkingConfig(thinking_budget=0)
        case Model.GEMINI_2_5_PRO:
            return ThinkingConfig(thinking_budget=128, include_thoughts=False)
        case Model.GEMINI_3_1_FLASH_LITE | Model.GEMINI_3_FLASH:
            return ThinkingConfig(thinking_level=ThinkingLevel.MINIMAL)
        case Model.GEMINI_3_1_PRO:
            return ThinkingConfig(thinking_level=ThinkingLevel.LOW)
        case _:
            return None  # Default (dynamic thinking is generally enabled)


def check_client_for_model(model: Model) -> genai.Client:
    if (
        model.value.endswith("-preview")
        and client.vertexai
        and client._api_client.location != "global"
    ):
        # Preview models are only available on the "global" location
        return genai.Client(location="global")

    return client


def get_retrier() -> tenacity.Retrying:
    return tenacity.Retrying(
        stop=tenacity.stop_after_attempt(7),
        wait=tenacity.wait_incrementing(start=10, increment=1),
        retry=should_retry_request,
        reraise=True,
    )


def should_retry_request(retry_state: tenacity.RetryCallState) -> bool:
    if not retry_state.outcome:
        return False
    err = retry_state.outcome.exception()
    if not isinstance(err, ClientError):
        return False
    print(f"❌ ClientError {err.code}: {err.message}")

    retry = False
    match err.code:
        case 400 if err.message is not None and " try again " in err.message:
            # Workshop: project accessing Cloud Storage for the first time (service agent provisioning)
            retry = True
        case 429:
            # Workshop: temporary project with 1 QPM quota
            retry = True
    print(f"🔄 Retry: {retry}")

    return retry


PRINT_COLUMNS = 80
PRINT_SEPARATOR_CHAR = "-"
PRINT_SEPARATOR = PRINT_COLUMNS * PRINT_SEPARATOR_CHAR


def print_caption(caption: str) -> None:
    print(f" {caption} ".center(PRINT_COLUMNS, PRINT_SEPARATOR_CHAR))


def print_separator() -> None:
    print(PRINT_SEPARATOR)


def display_response_info(response: GenerateContentResponse) -> None:
    if usage_metadata := response.usage_metadata:
        if usage_metadata.prompt_token_count:
            print(f"Input tokens    : {usage_metadata.prompt_token_count:9,d}")
        if usage_metadata.cached_content_token_count:
            print(f"Cached tokens   : {usage_metadata.cached_content_token_count:9,d}")
        if usage_metadata.candidates_token_count:
            print(f"Output tokens   : {usage_metadata.candidates_token_count:9,d}")
        if usage_metadata.thoughts_token_count:
            print(f"Thoughts tokens : {usage_metadata.thoughts_token_count:9,d}")
    if not response.candidates:
        print("❌ No `response.candidates`")
        return
    if (finish_reason := response.candidates[0].finish_reason) != FinishReason.STOP:
        print(f"❌ {finish_reason = }")
    if not response.text:
        print("❌ No `response.text`")
        return


def display_prompt(
    contents: list[str | Part],
    show_as: ShowAs,
    only_show_prompt: bool,
) -> None:
    def yield_prompt_strings() -> Iterator[str]:
        for content in contents:
            if isinstance(content, Part):
                yield f"{content!r}\n"
                continue
            yield content

    if only_show_prompt and show_as == ShowAs.DONT_SHOW:
        show_as = ShowAs.TEXT
    if show_as == ShowAs.DONT_SHOW:
        return

    separator = "\n" if show_as == ShowAs.MARKDOWN else ""
    prompt = separator.join(yield_prompt_strings())
    print_caption("Prompt")
    match show_as:
        case ShowAs.TEXT:
            print(prompt)
        case ShowAs.MARKDOWN:
            display_markdown(prompt)
    if only_show_prompt:
        print_separator()


def display_request_header(model_id: str, source: Source | str | None = None) -> None:
    print_caption(f"Request / {model_id}")


def display_response(response: GenerateContentResponse, show_as: ShowAs) -> None:
    if show_as == ShowAs.DONT_SHOW or not (response_text := response.text):
        return
    print_caption("Start of Response")
    response_text = response_text.strip()
    match show_as:
        case ShowAs.TEXT:
            print(response_text)
        case ShowAs.MARKDOWN:
            display_markdown(response_text)
    print_caption("End of Response")


def display_markdown(markdown: str) -> None:
    IPython.display.display(IPython.display.Markdown(markdown))


print("✅ Helpers defined")

---

## 🏗️ Prototyping


To start prototyping, let's define a short text of a few sentences:


In [ ]:
text = """
- Henry Walton Jones is a famous archaeologist. He is actually a "Junior" because he is named after his father.
- Sophie is Henry's daughter and shares his name. She is a software engineer.
- Bill is a rocket scientist and has been Sophie's lifelong friend. He has a dog named Beau. His full name is William Smith.
- Short Round was a teenager when he met Henry. They quickly became friends. Henry has just adopted him.
"""

---

### 📋 Listing characters


🧪 Let's see if Gemini can spot our characters…


In [ ]:
prompt = """
Using only the input data, list all persons and animals mentioned.
"""
generate_content(prompt, text)

---

### 📋 Listing characters & relationships


🧪 Now, let's see if it can connect the dots and figure out who is who…


In [ ]:
prompt = """
Using only the input data, list all persons and animals mentioned, and how they relate to each other.
"""
generate_content(prompt, text)

---

### 📚 Domain terminology


We're not domain experts on the problem we're trying to solve (yet!).

An LLM processes instructions based on the given prompt and its training knowledge. This knowledge is part of its long-term memory and we can learn a lot directly from the model itself.

🧪 Let's ask Gemini:


In [ ]:
prompt = """
What is the terminology used when building a knowledge graph?
Please provide a simple data example in JSON.
"""
generate_content(prompt)

---

## ⛏️ Tabular extraction


### 💠 Entities


🧪 Let's extract entities first:


In [ ]:
prompt = """
**Data Schema**

Entity:
- `id`: Unique integer identifier (0, 1, 2, …).
- `name`: Most complete name of the entity.
- `label`: `person`|`animal`.

**Instructions**

- Extract every distinct entity from the input data that fits an allowed `label`. Include named entities that are explicitly mentioned as well as those directly implied from context.
- Output the results as a JSON array inside a fenced code block.
"""

generate_content(prompt, text)

---

### 🔗 Relationships


🧪 Now, let's extract both the entities and their relationships:


In [ ]:
prompt = """
**Data Schema**

- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Most complete name of the entity.
  - `label`: `person`|`animal`.
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: `id` of the subject entity.
  - `link`: `snake_case` predicate describing the relationship.
  - `target_id`: `id` of the object entity.

**Instructions**

- Extract every distinct entity from the input data that fits an allowed `label`.
  - Include named entities that are explicitly mentioned as well as those directly implied from context.
- Extract every relationship between them.
  - If two entities share multiple distinct relationships, create a separate entry for each relationship.
- Output a JSON object with keys `entities` and `relationships` inside a fenced code block.
"""

response = generate_content(prompt, text, return_response=True)

---

### 💎 Output optimization


The JSON format has industry-wide support and is a native or intermediate format in many use cases. However, it is a pretty verbose format, definitely not optimized for size, as all keys are repeated for each object instance.

With LLMs, once the first token is generated, the rest of the generation time is roughly proportional to the number of output tokens. Similarly, the cost of a request is based on token usage (input + output), with output tokens being significantly more expensive than input tokens. In other words, a better output structure will have a positive impact on both the speed and cost of each request.

The problem we're solving can be seen as a tabular extraction, so it clearly calls for table outputs. An interesting possibility is to ask for Tab-Separated Values (TSV) outputs. For example, we can define our output to be formatted as 2 TSV tables:

````markdown
```tsv filename="entities.tsv"
id{TAB}name{TAB}type
[rows]
```

```tsv filename="relationships.tsv"
source_id{TAB}link{TAB}target_id
[rows]
```
````


Let's define some helpers to compare JSON and TSV outputs:


In [ ]:
# @title {display-mode: "form"}
import csv
import io
import json
import re
from typing import Literal


def dict_from_response(response: GenerateContentResponse) -> tuple[str, dict]:
    response_text = response.text or ""
    pattern = r"```json\s*(.*?)\s*```"
    match = re.search(pattern, response_text, re.DOTALL)
    json_str = match.group(1) if match else response_text
    try:
        data = json.loads(json_str)
        if not isinstance(data, dict):
            print("❌ Returning empty dict (could not parse response as dict)")
            data = {}
    except json.JSONDecodeError:
        print("❌ Returning empty dict (failed parsing the JSON string)")
        data = {}
    return response_text, data


def tsv_strings_from_dict(data: dict) -> str:
    output = ""
    for key, items in data.items():
        rows = ""
        if items:
            output_buffer = io.StringIO()
            headers = list(items[0].keys())
            writer = csv.DictWriter(output_buffer, fieldnames=headers, delimiter="\t")
            writer.writeheader()
            writer.writerows(items)
            rows = output_buffer.getvalue()
            output_buffer.close()
        output += f'```tsv filename="{key}.tsv"\n{rows}```\n'
    return output


def compare_json_response_vs_equivalent_tsv_sizes(
    response: GenerateContentResponse | None,
) -> None:
    def get_gain(rows: list[tuple[int, int]], col: Literal[0, 1]) -> str:
        val_0, val_1 = rows[0][col], rows[1][col]
        gain = 1 - val_1 / val_0
        return f"**{gain:.0%}**" if val_0 > 0 else "?"

    def yield_table_string_rows(rows: list[tuple[int, int]]) -> Iterator[list[str]]:
        yield ["", "Chars", "Tokens"]
        yield ["-", "-:", "-:"]
        for caption, values in zip(["JSON", "TSV"], rows):
            yield [caption, *[str(value) for value in values]]
        yield ["**Gain**", get_gain(rows, 0), get_gain(rows, 1)]

    def get_tsv_md(tsv_text: str) -> str:
        return "\n".join(["````markdown", tsv_text.strip(), "````"])

    def get_table_md(json_text: str, tsv_text: str) -> str:
        rows: list[tuple[int, int]] = []
        model = Model.DEFAULT
        model_id = model.value
        client = check_client_for_model(model)
        for s in [json_text, tsv_text]:
            chars = len(s)
            tokens = client.models.count_tokens(model=model_id, contents=s).total_tokens
            rows.append((chars, tokens or 0))
        return "\n".join(
            "| " + " | ".join(row) + " |" for row in yield_table_string_rows(rows)
        )

    if response is None:
        print("❌ response is None")
        return
    json_text, data = dict_from_response(response)
    tsv_text = tsv_strings_from_dict(data)

    print_caption("JSON → TSV Blocks")
    display_markdown(get_tsv_md(tsv_text))
    print_caption(" JSON → TSV Gain")
    display_markdown(get_table_md(json_text, tsv_text))
    print_separator()


print("✅ JSON vs TSV helpers defined")

🧪 And now, let's compare the JSON vs TSV token numbers for our latest request:


In [ ]:
compare_json_response_vs_equivalent_tsv_sizes(response)

💡 With a double-digit gain in the number of output tokens, our requests will be significantly faster (and cheaper)!


> ℹ️ Generating TSV outputs should work without any problem. We will need to be explicit about what's expected.


Now, let's finalize our code with structured and optimized data…


---

## 🚀 Finalization


### 🧩 Structure


First, it can be useful to define a structured prompt template, so we can focus on specific parts of our solution in a divide-and-conquer way.

Here's a possible prompt template:


In [ ]:
KNOWLEDGE_GRAPH_PROMPT_TEMPLATE = """
**Data Schema**

{data_schema}

**Instructions**

{instructions}

**Output Format**

{output_format}
"""

Then, here are possible `Entity`, `Relationship`, and `KnowledgeGraph` data classes with the matching output format:


In [ ]:
from dataclasses import dataclass, field


@dataclass
class Entity:
    id: int
    name: str
    label: str


@dataclass
class Relationship:
    source_id: int
    link: str
    target_id: int


@dataclass
class KnowledgeGraph:
    entities: list[Entity] = field(default_factory=list)
    relationships: list[Relationship] = field(default_factory=list)


TAB = "\t"
KNOWLEDGE_GRAPH_OUTPUT_FORMAT = f"""
```tsv filename="entities.tsv"
id{TAB}name{TAB}label
[rows]
```

```tsv filename="relationships.tsv"
source_id{TAB}link{TAB}target_id
[rows]
```
"""

> ℹ️ If you use multiple entity or relationship dataclasses in your solution, the output format specification can be dynamically generated using `dataclasses` features.


Now, let's define a few helpers to generate a knowledge graph:


In [ ]:
# @title {display-mode: "form"}
from dataclasses import fields, is_dataclass
from typing import get_args, get_origin, get_type_hints


def generate_knowledge_graph(
    data_schema: str,
    instructions: str,
    source: Source | str,
    *,
    model: Model | None = None,
    config: GenerateContentConfig | None = None,
    system_instruction: str | None = None,
    show_prompt: ShowAs = ShowAs.DONT_SHOW,
    show_response: ShowAs = ShowAs.DONT_SHOW,
) -> KnowledgeGraph:
    prompt = get_prompt_for_data_schema_and_instructions(data_schema, instructions)
    response = generate_content(
        prompt,
        source,
        model=model,
        config=config,
        system_instruction=system_instruction,
        show_prompt=show_prompt,
        show_response=show_response,
        return_response=True,
    )

    knowledge_graph = (
        parse_list_dataclass(KnowledgeGraph, response)
        if isinstance(response, GenerateContentResponse)
        else KnowledgeGraph()
    )
    display_knowledge_graph_info(knowledge_graph)

    return knowledge_graph


def show_knowledge_graph_prompt(
    data_schema: str,
    instructions: str,
    source: Source | str,
    *,
    model: Model | None = None,
    config: GenerateContentConfig | None = None,
    system_instruction: str | None = None,
    show_as: ShowAs = ShowAs.TEXT,
) -> None:
    prompt = get_prompt_for_data_schema_and_instructions(data_schema, instructions)
    generate_content(
        prompt,
        source,
        model=model,
        config=config,
        system_instruction=system_instruction,
        show_prompt=show_as,
        only_show_prompt=True,
    )


def get_prompt_for_data_schema_and_instructions(
    data_schema: str,
    instructions: str,
) -> str:
    return KNOWLEDGE_GRAPH_PROMPT_TEMPLATE.format(
        data_schema=data_schema.strip(),
        instructions=instructions.strip(),
        output_format=KNOWLEDGE_GRAPH_OUTPUT_FORMAT.strip(),
    ).strip()


def parse_list_dataclass[T](cls: type[T], response: GenerateContentResponse) -> T:
    assert is_dataclass(cls)
    if not (response_text := response.text):
        return cls()

    data = {}
    for f in fields(cls):
        origin, list_types = get_origin(f.type), get_args(f.type)
        assert origin is list and len(list_types) == 1
        data[f.name] = parse_tsv_block(list_types[0], response_text, f.name)
    if not data:
        print("❌ Could not parse the response")

    return cls(**data)


def parse_tsv_block[T](cls: type[T], data: str, tsv_filestem: str) -> list[T]:
    rows = []
    valid_fields = get_type_hints(cls)
    tsv_string = extract_tsv_block(data, tsv_filestem)
    for row in csv.DictReader(io.StringIO(tsv_string), delimiter="\t"):
        casted_data = {}
        for key, value in row.items():
            if key not in valid_fields or value is None:
                continue
            field_type = valid_fields[key]
            try:  # Note: Works only for directly castable types such as int, float, str, enum (e.g., not bool)
                casted_data[key] = field_type(value)
            except (ValueError, TypeError):
                print(f'❌ Could not cast "{value}" to {field_type} → Skipping {row=}')
                break
        else:
            rows.append(cls(**casted_data))

    return rows


def extract_tsv_block(data: str, filestem: str) -> str:
    pattern = rf'```tsv filename="{re.escape(filestem)}.tsv"\s*\n(.*?)```'
    if not (match := re.search(pattern, data, flags=re.DOTALL)):
        print(f'❌ Could not find a TSV block for "{filestem=}"')
        return ""
    return match.group(1)


def display_knowledge_graph_info(kg: KnowledgeGraph) -> None:
    print_caption("Knowledge Graph Info")
    print(f"Entities      : {len(kg.entities):3,d}")
    print(f"Relationships : {len(kg.relationships):3,d}")
    print_separator()


print("✅ Knowledge graph generation helpers defined")

And here is a possible data schema with some instructions to generate a knowledge graph for our book analysis use case:


In [ ]:
from enum import StrEnum, auto


class BookAnalysisEntityLabel(StrEnum):
    PERSON = auto()
    ANIMAL = auto()
    ORGANIZATION = auto()


def pipe_delimited_union(enum: type[StrEnum]) -> str:
    return "|".join(f"`{e.value}`" for e in enum)


BOOK_ANALYSIS_DATA_SCHEMA = f"""
- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Name of the entity.
  - `label`: {pipe_delimited_union(BookAnalysisEntityLabel)}.
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: `id` of the subject entity.
  - `link`: `snake_case` predicate describing the relationship.
  - `target_id`: `id` of the object entity.
"""

BOOK_ANALYSIS_INSTRUCTIONS = """
- Extract every distinct entity that fits an allowed `label`:
  - Include named entities that are explicitly mentioned as well as those directly implied from the context.
  - Assign each entity the most complete `name` available in, and only in, the input data.
- Extract every distinct relationship between them:
  - Capture key relationships belonging to the following categories (do not limit to these examples, define additional relationships as needed):
    - Kinship & Social Dynamics: `daughter_of`, `husband_of`, `fiancée_of`, `alias_of`, `friend_of`, `enemy_of`, …
    - Organizational & Professional Ties: `employer_of`, `colleague_of`, `ally_of`, …
    - Emotional & Psychological States: `in_love_with`, `fears`, `trusts`, …
    - Ownership, Control & Dependency: `owner_of`, `pet_of`, `controls`, …
    - Action, Conflict & Narrative Events: `defies_in_duel`, `saves`, `kills`, …
    - Communication & Information Exchange: `mentors`, `shares_secret_with`, `lies_to`, `extorts`, …
  - For every relationship, determine if an inverse or symmetric relationship is implied and include it as a separate entry.
    - Example 1: if A is `employer_of` B, then B is `employee_of` A.
    - Example 2: if A is `neighbor_of` B, then B is `neighbor_of` A.
  - If two entities share multiple distinct relationships, create a separate entry for each.
"""

Verify the structured prompt:


In [ ]:
show_knowledge_graph_prompt(
    BOOK_ANALYSIS_DATA_SCHEMA,
    BOOK_ANALYSIS_INSTRUCTIONS,
    text,
    show_as=ShowAs.TEXT,
)

🧪 Let's generate a knowledge graph:


In [ ]:
knowledge_graph = generate_knowledge_graph(
    BOOK_ANALYSIS_DATA_SCHEMA,
    BOOK_ANALYSIS_INSTRUCTIONS,
    text,
    show_response=ShowAs.TEXT,
)

print(knowledge_graph.entities)
print(knowledge_graph.relationships)

This is looking good!

Now, let's go to the next stage and build a network graph from our data…


---

### 🪢 Network graph


Now that we have our entities and relationships neatly packed in dataclasses, let's bring them to life. We'll use `networkx` to build a directed graph, calculate node centralities to spot the main characters, and use the Louvain method to detect communities (clusters of closely related entities)…


In [ ]:
# @title {display-mode: "form"}
import textwrap
from typing import cast

import networkx as nx
from networkx.algorithms.community.louvain import louvain_communities

# Wrap names over multiple lines (avoids long node labels)
MULTILINE_NAMES = True


def build_graph(kg: KnowledgeGraph, remove_orphan_nodes: bool) -> nx.DiGraph:
    graph = nx.DiGraph()

    # Nodes ← Entities
    entity_name_from_id: dict[int, str] = {}
    for entity in kg.entities:
        name = entity.name
        if MULTILINE_NAMES:
            name = "\n".join(textwrap.wrap(name, width=10))
        entity_name_from_id[entity.id] = name
        graph.add_node(name, name=name)

    # Edges ← Relationships
    for relationship in kg.relationships:
        source_node = entity_name_from_id.get(relationship.source_id, "")
        target_node = entity_name_from_id.get(relationship.target_id, "")
        if not source_node or not target_node:
            print(f"❌ Skipping empty node:\n{source_node=}\n{target_node=}")
            continue
        # For the sake of simplicity, each link has the same weight
        # For better community detection, define your own weight mappings (e.g., family members have stronger bonds)
        weight = 1
        edge_label = relationship.link
        if graph.has_edge(source_node, target_node):
            existing_data = graph[source_node][target_node]
            existing_data["link"] += f"\n{edge_label}"
            existing_data["weight"] += weight
        else:
            graph.add_edge(source_node, target_node, link=edge_label, weight=weight)

    if remove_orphan_nodes and graph.nodes():
        orphan_nodes = list(nx.isolates(graph))
        if orphan_nodes:
            graph.remove_nodes_from(orphan_nodes)

    return graph


NODE_CENTRALITY = "node_centrality"
NODE_COMMUNITY_INDEX = "node_community_index"
NODE_COLOR = "node_color"
EDGE_COLOR = "edge_color"


def color_gen(color_count: int) -> Iterator[str]:
    B50, R50, Y50, G50 = ("#4285F4", "#EA4335", "#FBBC04", "#34A853")
    B20, R20, Y20, G20 = ("#AECBFA", "#F6AEA9", "#FDE293", "#A8DAB5")
    B05, R05, Y05, G05 = ("#E8F0FE", "#FCE8E6", "#FEF7E0", "#E6F4EA")
    COLORS = [B50, R50, Y50, G50, B20, R20, Y20, G20, B05, R05, Y05, G05]
    for i in range(color_count):
        yield COLORS[i % len(COLORS)]


Positions = dict
Node = str
Community = set[Node]
Communities = list[Community]
Nodes = list[Node]


def init_graph_data(graph: nx.Graph) -> Nodes:
    # Node centralities and communities
    centralities = nx.betweenness_centrality(graph, endpoints=True)
    communities = cast(Communities, louvain_communities(graph, seed=42))

    # Community colors
    community_count = len(communities)
    community_colors = list(color_gen(community_count))

    # Node centralities, indices, and colors
    for community_index, community in enumerate(communities):
        for node_key in community:
            node = graph.nodes[node_key]
            node[NODE_CENTRALITY] = centralities[node_key]
            node[NODE_COMMUNITY_INDEX] = community_index
            node[NODE_COLOR] = community_colors[community_index]

    # Edge colors
    for node_key_i, node_key_j, edge_data in graph.edges(data=True):
        node_i = graph.nodes[node_key_i]
        node_j = graph.nodes[node_key_j]
        same_community = node_i[NODE_COMMUNITY_INDEX] == node_j[NODE_COMMUNITY_INDEX]
        edge_data[EDGE_COLOR] = node_i[NODE_COLOR] if same_community else "#8888"

    return nodes_sorted_by_community(graph, communities)


def nodes_sorted_by_community(graph: nx.Graph, communities: Communities) -> Nodes:
    def node_centrality(node_key):
        return graph.nodes[node_key][NODE_CENTRALITY]

    entities = []
    for community in communities:
        entities.extend(sorted(community, key=node_centrality, reverse=True))

    return entities


def compute_community_graph_and_node_positions(
    graph: nx.DiGraph,
    entities: Nodes,
) -> tuple[nx.Graph, Positions]:
    undirected_graph = nx.Graph()
    for entity in entities:
        undirected_graph.add_node(entity, **graph.nodes[entity])

    for u, v, data in graph.edges(data=True):
        weight = data.get("weight", 1)
        if undirected_graph.has_edge(u, v):
            undirected_graph[u][v]["weight"] += weight
        else:
            undirected_graph.add_edge(u, v, **data)

    positions = nx.circular_layout(undirected_graph)
    positions = nx.arf_layout(undirected_graph, positions, seed=42)

    return undirected_graph, positions


@dataclass
class GraphData:
    knowledge_graph: KnowledgeGraph
    remove_orphan_nodes: bool = True
    graph: nx.DiGraph = field(init=False)
    nodes: Nodes = field(init=False)
    community_graph: nx.Graph = field(init=False)
    positions: Positions = field(init=False)

    def __post_init__(self) -> None:
        self.graph = build_graph(self.knowledge_graph, self.remove_orphan_nodes)
        self.nodes = init_graph_data(self.graph)
        self.community_graph, self.positions = (
            compute_community_graph_and_node_positions(self.graph, self.nodes)
        )


print("✅ Network graph helpers defined")

Let's test this:


In [ ]:
graph_data = GraphData(knowledge_graph)

print(f"{graph_data.graph = !s}")
print(f"{graph_data.nodes = }")

---

### 🎨 Data visualization


A graph is much easier to understand when you can actually see it! We can use `matplotlib` to draw our network graph. We'll size the nodes based on their centrality and color them by community. To make it even easier to digest, we'll generate an animated sequence highlighting each character's connections one by one…


In [ ]:
# @title {display-mode: "form"}
import typing
from io import BytesIO

import matplotlib.pyplot as plt
from IPython import display
from PIL import Image as PilImage
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure


class AnimationFormat(StrEnum):
    # Matches PIL.Image.Image.format
    WEBP = auto()
    PNG = auto()
    GIF = auto()


ANIMATION_INTRO_DURATION = 2000
ANIMATION_FRAME_DURATION = 500
FOCUSED_NODE_BORDER_COLOR = "#F0F8"
EDGE_STYLE = "arc3,rad=0.2"
NodeSizes = dict[Node, int]

if running_in_colab_env():
    ANIMATION_FORMAT = AnimationFormat.WEBP
else:
    ANIMATION_FORMAT = AnimationFormat.GIF


def init_figure(title: str) -> Figure:
    factor = 1
    fig = plt.figure(figsize=(16 * factor, 9 * factor))

    subtitle = "Knowledge Graph"
    plt.title(title, loc="left")
    plt.title(subtitle, loc="right")
    plt.axis("off")
    plt.tight_layout(pad=2)

    return fig


def draw_nodes(graph: nx.Graph, positions: Positions) -> NodeSizes:
    node_view = graph.nodes(data=True)
    node_sizes = {
        node: max(500, int(15000 * data[NODE_CENTRALITY])) for node, data in node_view
    }
    node_colors = [str(data[NODE_COLOR]) for _, data in node_view]
    border_width = 3.0

    nx.draw_networkx_nodes(
        graph,
        pos=positions,
        node_size=list(node_sizes.values()),
        node_color=node_colors,
        alpha=0.95,
        linewidths=border_width,
    )
    nx.draw_networkx_labels(graph, positions)

    # Extend axis limits (default rounding changes them too much otherwise)
    ax = plt.gca()
    (x_min, x_max), (y_min, y_max) = ax.get_xlim(), ax.get_ylim()
    x_min, x_max = int(x_min - 1.0), int(x_max + 1.0)
    y_min, y_max = int(y_min - 1.0), int(y_max + 1.0)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    return node_sizes


def draw_edges(
    graph: nx.Graph,
    positions: Positions,
    focused_node: Node | None,
    node_sizes: NodeSizes,
) -> None:
    if focused_node:
        edgelist = graph.edges([focused_node], data=True)
    else:
        edgelist = graph.edges(data=True)
    edge_colors = [data[EDGE_COLOR] for _, _, data in edgelist]

    edge_style = ":" if focused_node else "-"
    arrow_style = "-|>" if focused_node else "-"
    ordered_sizes = [node_sizes[n] for n in graph.nodes()]
    nx.draw_networkx_edges(
        graph,
        positions,
        list(edgelist),
        edge_color=edge_colors,
        style=edge_style,
        alpha=0.9,
        arrows=True,
        arrowstyle=arrow_style,
        arrowsize=20,
        connectionstyle=EDGE_STYLE,
        node_size=ordered_sizes,
    )

    if not focused_node:
        return

    edge_labels = {(u, v): data["link"] for u, v, data in edgelist}
    nx.draw_networkx_edge_labels(
        graph,
        positions,
        edge_labels,
        font_size=8,
        font_family="monospace",
        bbox=dict(ec="#FFF8", fc="#FFF8"),
        node_size=ordered_sizes,  # type: ignore
        connectionstyle=EDGE_STYLE,  # type: ignore
    )


def display_graph(title: str, graph_data: GraphData) -> None:
    init_figure(title)

    node_graph = graph_data.graph
    edge_graph = graph_data.graph
    positions = graph_data.positions
    node_sizes = draw_nodes(node_graph, positions)

    for focused_node in graph_data.nodes:
        draw_edges(edge_graph, positions, focused_node, node_sizes)

    plt.show()


def yield_images(title: str, graph_data: GraphData) -> Iterator[PilImage.Image]:
    fig = init_figure(title)

    positions = graph_data.positions
    node_graph = graph_data.community_graph
    node_sizes = draw_nodes(node_graph, positions)
    edge_graph = graph_data.graph

    for focused_node in [None, *graph_data.nodes]:
        if focused_node:
            draw_edges(edge_graph, positions, focused_node, node_sizes)

        canvas = cast(FigureCanvasAgg, fig.canvas)
        canvas.draw()
        image_size = canvas.get_width_height()
        image_bytes = canvas.buffer_rgba()
        yield PilImage.frombytes("RGBA", image_size, image_bytes).convert("RGB")


def generate_animation(
    title: str,
    graph_data: GraphData,
    format: AnimationFormat,
) -> BytesIO:
    frames = list(yield_images(title, graph_data))
    assert len(frames) >= 1

    if format == AnimationFormat.GIF:
        # Dither all frames with the same palette to optimize the animation
        # The animation is cumulative, so most colors are in the last frame
        method = PilImage.Quantize.MEDIANCUT
        palettized = frames[-1].quantize(method=method)
        frames = [frame.quantize(method=method, palette=palettized) for frame in frames]

    # The animation will be played in a loop: start cycling with the most complete frame
    first_frame = frames[-1]
    next_frames = frames[:-1]
    durations = [ANIMATION_INTRO_DURATION]
    durations += [ANIMATION_FRAME_DURATION] * len(next_frames)
    params: dict[str, typing.Any] = dict(
        save_all=True,
        append_images=next_frames,
        duration=durations,
        loop=0,
    )
    match format:
        case AnimationFormat.GIF:
            params.update(optimize=False)
        case AnimationFormat.PNG:
            params.update(optimize=True)
        case AnimationFormat.WEBP:
            params.update(lossless=True)

    image_io = BytesIO()
    first_frame.save(image_io, str(format), **params)

    return image_io


def display_graph_animation(title: str, graph_data: GraphData) -> None:
    image_io = generate_animation(title, graph_data, ANIMATION_FORMAT)
    plt.close()  # Don't display the matplotlib image

    image_bytes = image_io.getvalue()
    ipython_image = display.Image(image_bytes)
    display.display(ipython_image)


def display_knowledge_graph(
    source: Source | str,
    knowledge_graph: KnowledgeGraph,
    animated: bool = False,
    remove_orphan_nodes: bool = True,
) -> None:
    if not knowledge_graph.entities:
        return
    graph_data = GraphData(knowledge_graph, remove_orphan_nodes)
    title = source.name if isinstance(source, Source) else f"{source[:50]}…"
    if animated:
        display_graph_animation(title, graph_data)
    else:
        display_graph(title, graph_data)


def extract_knowledge_graph(
    data_schema: str,
    instructions: str,
    source: Source | str,
    model: Model | None = None,
    animated: bool = False,
) -> None:
    knowledge_graph = generate_knowledge_graph(
        data_schema,
        instructions,
        source,
        model=model,
    )
    display_knowledge_graph(source, knowledge_graph, animated)


print("✅ Data visualization helpers defined")

Let's test this:


In [ ]:
display_knowledge_graph(text, knowledge_graph, animated=True)

💡 Remarks

- While this simple approach is great for a quick overview, you might want to swap it out for more specialized libraries if you want to explore the graph interactively.
- Likewise, if you extract thousands of entities from a massive document, your graph will quickly turn into an unreadable hairball. For larger datasets, you'll want to export your nodes and edges to a dedicated graph database.


---

## ✅ Challenge completed


In [ ]:
def analyze_book(
    source: Source,
    model: Model | None = None,
    animated: bool = False,
) -> None:
    extract_knowledge_graph(
        BOOK_ANALYSIS_DATA_SCHEMA,
        BOOK_ANALYSIS_INSTRUCTIONS,
        source,
        model,
        animated,
    )

In [ ]:
analyze_book(Classic.fr_zola_thérèse_raquin)

In [ ]:
analyze_book(Classic.en_hugo_les_misérables)

In [ ]:
analyze_book(Classic.fr_dumas_comte_de_monte_cristo_1)

In [ ]:
analyze_book(Collection.fr_comte_de_monte_cristo, animated=True)

---

## 🔭 Generalizing to other contents


A legal agreement is one of the densest types of documents. It is generally composed of many articles and clauses, with every sentence carrying specific legal weight, obligations, or definitions. Thanks to Gemini's 1M token context, feeding a 100-page PDF in a single request is a breeze, but if we just ask for a generic extraction, the model will naturally summarize and miss the granular details we actually care about.


🧪 Let's test a minimalist, fully open prompt:


In [ ]:
source = Document.en_pharma_dev_agreement
model = Model.GEMINI_3_FLASH

AGREEMENT_OPEN_DATA_SCHEMA = """
- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Name of the entity.
  - `label`
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: `id` of the subject entity.
  - `link`: `snake_case` predicate describing the relationship.
  - `target_id`: `id` of the object entity.
"""

AGREEMENT_OPEN_INSTRUCTIONS = """
Perform an entity and relationship extraction.
"""

extract_knowledge_graph(
    AGREEMENT_OPEN_DATA_SCHEMA,
    AGREEMENT_OPEN_INSTRUCTIONS,
    source,
    model,
)

💡 Remarks

- Notice how we just passed a PDF directly? Gemini processes the document natively, automatically extracting text and images, and managing OCR along the way.
- This minimalist prompt has the most degrees of freedom. It is impossible to guess which data should be extracted and how it will be useful for your use case. As a result, only high-level entities and relationships are extracted, which provides a nice summary of the agreement.
- LLMs are trained to synthesize information. With such fully open instructions, the default behavior proves valuable for avoiding the generation of many unnecessary tokens.


🧪 Then, let's try semi-open instructions focusing on legal obligations:


In [ ]:
class AgreementEntityLabel(StrEnum):
    ROLE = auto()
    PERSON = auto()
    ORGANIZATION = auto()
    JURISDICTION = auto()
    LOCATION = auto()
    FINANCIAL_AMOUNT = auto()
    DATE = auto()
    ASSET = auto()
    PRODUCT = auto()


AGREEMENT_SEMI_OPEN_DATA_SCHEMA = f"""
- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Name of the entity.
  - `label`: {pipe_delimited_union(AgreementEntityLabel)}.
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: The `id` of the subject entity.
  - `link`: `snake_case` predicate describing the relationship.
  - `target_id`: The `id` of the object entity.
"""

AGREEMENT_SEMI_OPEN_INSTRUCTIONS = """
- Extract every distinct entity from the input data matching an allowed entity `label`.
- Extract every distinct relationship representing a legal obligation between them.
- Granularity: You must be highly granular. If multiple distinct relationships exist between the same pair of entities, create a separate entry for each.
- Exhaustiveness: You must be exhaustive. Process the entire document. Do not omit valid entities or relationships. You have no output token limit.
"""

extract_knowledge_graph(
    AGREEMENT_SEMI_OPEN_DATA_SCHEMA,
    AGREEMENT_SEMI_OPEN_INSTRUCTIONS,
    source,
    model,
)

💡 Remarks

- This semi-open prompt lists the types of entities to extract, which naturally increases the entity count and specificity. By also explicitly demanding exhaustiveness, we counter the model's natural summarization bias.
- The resulting graph is denser and reflects the legal complexity of the document, rather than just giving us a high-level summary.
- The extracted relationships are still high-level though, due to the openness of this part in our prompt.


What if we don't care about the legal obligations, but rather the document's architecture? Let's shift our focus to the structure itself and extract how sections, clauses, and defined terms are hierarchically organized…

🧪 And now, let's check closed instructions focusing on the document structure:


In [ ]:
class AgreementStructureEntityLabel(StrEnum):
    DEFINED_TERM = auto()
    DOCUMENT_SECTION = auto()
    DOCUMENT = auto()


class AgreementStructureRelationshipType(StrEnum):
    DEFINED_IN = auto()
    CONTAINS = auto()


AGREEMENT_STRUCTURAL_DATA_SCHEMA = f"""
- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Name of the entity.
  - `label`: {pipe_delimited_union(AgreementStructureEntityLabel)}.
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: The `id` of the subject entity.
  - `link`: {pipe_delimited_union(AgreementStructureRelationshipType)}.
  - `target_id`: The `id` of the object entity.
"""

AGREEMENT_STRUCTURAL_OPEN_INSTRUCTIONS = """
- Extract every distinct entity matching an allowed `label`.
- Extract every distinct relationship representing a structural connection (hierarchical organization) between these entities.
- Granularity: You must be comprehensive and highly granular. If multiple distinct relationships exist between the same pair of entities, create a separate entry for each.
"""

extract_knowledge_graph(
    AGREEMENT_STRUCTURAL_DATA_SCHEMA,
    AGREEMENT_STRUCTURAL_OPEN_INSTRUCTIONS,
    source,
    model,
)

---

## 💡 Analyze your own contents


In [ ]:
class MyDocs(Source):
    # Classic books
    en_lewis_carroll_alice = project_gutenberg_txt_url(11)
    de_goethe_die_leiden_des_jungen_werther = project_gutenberg_txt_url(2407)
    de_fontane_effi_briest = project_gutenberg_txt_url(5323)

    # Local files
    # local_file = local_file("my_file.ext")

    # Online files
    # online_file = "https://path/to/my_file.ext"


MY_DATA_SCHEMA = """
- Entity:
  - `id`: Unique integer identifier (0, 1, 2, …).
  - `name`: Name of the entity.
  - `label`
- Relationship: Connection from a source entity to a target entity.
  - `source_id`: `id` of the subject entity.
  - `link`: `snake_case` predicate describing the relationship.
  - `target_id`: `id` of the object entity.
"""

MY_INSTRUCTIONS = """
Perform an entity and relationship extraction.
"""

In [ ]:
# analyze_book(MyDocs.en_lewis_carroll_alice)

# extract_knowledge_graph(MY_DATA_SCHEMA, MY_INSTRUCTIONS, MyDocs.local_file)

---

## 🏁 Conclusion


We managed to extract and build knowledge graphs from documents with the following steps:

- Prototyping with open prompts to develop intuition on Gemini's natural strengths
- Crafting increasingly specific prompts using a tabular extraction strategy
- Generating structured outputs to move towards production-ready code
- Adding data visualization for easier interpretation of responses and smoother iterations
- Adapting default parameters to optimize the results
- Conducting more tests, iterating, and even enriching the extracted data

These principles should apply to many other data extraction domains and allow you to solve your own complex problems. Have fun and happy solving!


---

## ➕ More!

- Explore typical use cases in the [Agent Platform Prompt Gallery](https://console.cloud.google.com/agent-platform/studio/prompt-gallery)
- Stay updated with the [Agent Platform Release Notes](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/release-notes)
